# BS49903 · DS/AI for Biology
## Assignment 2

**Due: Friday, November 13, 11:59 PM**

**Instructions.** Work through the exercises in order. Fill in every cell marked `#### YOUR CODE HERE ####` and answer the written questions in the markdown cells provided. You may use any publicly available Python package unless instructed otherwise. Do not delete the cells that are given to you.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.datasets import load_breast_cancer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans

%matplotlib inline

-----
## Part 1 — PCA and k-means on the Breast Cancer Wisconsin dataset

The Breast Cancer Wisconsin dataset contains **569 biopsies**. For each biopsy, **30 features** were measured from the cell nuclei in a digitised image (mean radius, mean texture, mean concavity, …). Each biopsy is labelled **malignant** (0) or **benign** (1).

In this part you will implement the two unsupervised methods from the lectures — **PCA** and **k-means** — *from scratch* with NumPy, and check your implementations against `scikit-learn`. The labels are only used at the very end, to see whether the structure we find without them is biologically meaningful.

Run the cell below to load the data.

In [ ]:
data = load_breast_cancer()
X_raw = data.data          # shape (569, 30)
y = data.target            # 0 = malignant, 1 = benign
feature_names = data.feature_names

print("X_raw shape:", X_raw.shape)
print("Number of malignant / benign:", (y == 0).sum(), "/", (y == 1).sum())
pd.DataFrame(X_raw, columns=feature_names).head()

### Exercise 1.1

Features in this dataset live on wildly different scales (`mean area` is in the hundreds, `mean fractal dimension` is around 0.06). PCA finds the directions of largest **variance**, so without rescaling the first component would simply point along `mean area`.

**Standardize** every feature so that it has mean 0 and standard deviation 1 across the 569 samples:

$$
x_{ij}^{\text{scaled}} = \frac{x_{ij} - \mu_j}{\sigma_j}
$$

**Implement this yourself with NumPy** (do not use `StandardScaler` to compute `X`), store the result in `X`, and then verify your result against `StandardScaler().fit_transform(X_raw)` by printing the maximum absolute difference between the two.

In [ ]:
#### YOUR CODE HERE ####

X = ...    # shape (569, 30), each column with mean 0 and std 1

X_sklearn = StandardScaler().fit_transform(X_raw)
print("Column means (first 5):", X.mean(axis=0)[:5].round(6))
print("Column stds  (first 5):", X.std(axis=0)[:5].round(6))
print("max |difference| vs StandardScaler:", np.abs(X - X_sklearn).max())

### Exercise 1.2

PCA can be computed directly from the **singular value decomposition** of the centred data matrix. Our `X` is already centred (every column has mean 0), so

$$
X = U \, S \, V^\top
$$

where the columns of $V$ are the **principal directions** (the loadings), and the singular values $s_1 \ge s_2 \ge \dots$ on the diagonal of $S$ tell us how much variance each direction carries:

$$
\text{variance along PC}_k = \frac{s_k^2}{n-1}, \qquad
\text{explained variance ratio}_k = \frac{s_k^2}{\sum_j s_j^2}
$$

The **scores** (the coordinates of every sample in the new axes) are the projection of the data onto the principal directions:

$$
Z = X V \;=\; U S
$$

NumPy computes the SVD with

```python
U, s, Vt = np.linalg.svd(X, full_matrices=False)   # s is a 1-D array of singular values, Vt = V^T
```

**Implement this yourself with NumPy** — do not use `sklearn.decomposition.PCA` in this exercise.

1. Compute the SVD of `X` with `np.linalg.svd`.
2. **Calculate the explained variance ratio yourself** from the singular values `s`, using the formula above, for **all 30** components and store it in `evr_svd`.
3. **Calculate the scores yourself** on the **first two** components, store them in `Z_svd` (shape `(569, 2)`), and store the corresponding two principal directions in `W_svd` (shape `(2, 30)`, one row per component).
4. Print the explained variance ratio of PC1 and PC2, and run the plotting code that follows.

In [ ]:
#### YOUR CODE HERE ####

U, s, Vt = ...

evr_svd = ...       # shape (30,), sums to 1
W_svd = ...         # shape (2, 30): first two principal directions
Z_svd = ...         # shape (569, 2): scores on PC1 and PC2

print("Explained variance ratio, PC1 and PC2:", evr_svd[:2].round(4))
print("Cumulative explained variance, first 10 PCs:", np.cumsum(evr_svd)[:10].round(3))

# plot the samples in the PC1 / PC2 plane
fig, ax = plt.subplots(figsize=(7, 6))
ax.scatter(Z_svd[:, 0], Z_svd[:, 1], alpha=0.5, s=15)
ax.set_xlabel(f"PC1 ({100 * evr_svd[0]:.1f}% of variance)")
ax.set_ylabel(f"PC2 ({100 * evr_svd[1]:.1f}% of variance)")
ax.set_title("PCA from SVD")
plt.show()

### Exercise 1.3

Now compare your result with `scikit-learn`:

```python
pca = PCA(n_components=2)
Z_sk = pca.fit_transform(X)
pca.components_                # shape (2, 30)
pca.explained_variance_ratio_  # shape (2,)
```

1. Fit `PCA(n_components=2)` on `X` and print its explained variance ratio next to yours.
2. Compare `pca.components_` with your `W_svd`, and `Z_sk` with your `Z_svd`, by printing the maximum absolute difference **for each component separately**.
3. In the markdown cell, explain what you observe. If a component does not match, is your implementation wrong? (*Hint: a principal direction $v$ and its negative $-v$ describe exactly the same axis.*)

In [ ]:
#### YOUR CODE HERE ####

pca = PCA(n_components=2)
Z_sk = ...

print("Explained variance ratio  scikit-learn:", ...)
print("Explained variance ratio  SVD:         ", ...)

for k in range(2):
    diff_W = ...   # max |difference| between the k-th principal direction from sklearn and from SVD
    diff_Z = ...   # max |difference| between the k-th score column from sklearn and from SVD
    print(f"PC{k+1}:  max |diff| directions = {diff_W:.2e}   max |diff| scores = {diff_Z:.2e}")

*Your answer here.*

### Exercise 1.4

Now **implement k-means yourself, from scratch with NumPy** — do not use `sklearn.cluster.KMeans` in this exercise. Recall the algorithm from the lecture:

1. **Initialize** $k$ centroids. Here, pick $k$ samples at random from the data (using `rng` so your result is reproducible).
2. **Assign** every sample to its nearest centroid (Euclidean distance over all 30 features).
3. **Update** each centroid to the mean of the samples assigned to it.
4. Repeat 2–3 until the assignments stop changing (or `max_iter` is reached).

The objective that k-means minimises is the **within-cluster sum of squares** (also called *inertia*):

$$
J = \sum_{i=1}^{n} \lVert x_i - c_{a(i)} \rVert^2
$$

where $a(i)$ is the cluster assigned to sample $i$ and $c_j$ is the centroid of cluster $j$.

Fill in the function `kmeans` below: **write the assignment step, the centroid update and the inertia $J$ yourself**. It must return the final labels (an integer array of length $n$), the final centroids (shape `(k, p)`), and the final value of $J$. Then run it on `X` with $k = 2$ and print $J$ and the number of samples in each cluster.

In [ ]:
rng = np.random.default_rng(seed=0)


def kmeans(X, k, rng, max_iter=100):
    """k-means clustering from scratch.

    Returns
    -------
    labels    : int array of shape (n,)   cluster id of every sample
    centroids : array of shape (k, p)     final centroid of every cluster
    inertia   : float                     within-cluster sum of squares J
    """
    n, p = X.shape

    # 1. initialize: pick k distinct samples as the starting centroids
    init_idx = rng.choice(n, size=k, replace=False)
    centroids = X[init_idx].copy()

    labels = np.full(n, -1)
    for it in range(max_iter):
        #### YOUR CODE HERE ####

        # 2. assign every sample to the nearest centroid
        new_labels = ...

        # stop if nothing changed
        if np.array_equal(new_labels, labels):
            break
        labels = new_labels

        # 3. move every centroid to the mean of its samples
        ...

    inertia = ...
    return labels, centroids, inertia


labels_mine, centroids_mine, J_mine = kmeans(X, k=2, rng=rng)

print("J (inertia):", round(J_mine, 2))
print("Cluster sizes:", np.bincount(labels_mine))

### Exercise 1.5

There is nothing to implement here. Run the cell below to check your k-means against `scikit-learn`'s `KMeans`. Your inertia should be very close to `scikit-learn`'s, and the agreement between the two should be close to 1 (small differences can come from the different starting centroids); if not, revisit Exercise 1.4. The plot shows the samples in the PC1/PC2 plane from Exercise 1.2, coloured by your clusters (left) and by the true malignant/benign labels (right).

**Question:** In the markdown cell below, answer: where in the PC1/PC2 plane do the k-means clusters disagree with the true labels, and why might that be?

In [ ]:
km = KMeans(n_clusters=2, n_init=10, random_state=0).fit(X)

print(f"Inertia  from scratch: {J_mine:.2f}")
print(f"Inertia  scikit-learn: {km.inertia_:.2f}")

# cluster ids are arbitrary (your 0 may be scikit-learn's 1), so take the better of the two matchings
def agreement(a, b):
    same = np.mean(a == b)
    return max(same, 1 - same)

print(f"Agreement, from scratch vs scikit-learn: {agreement(labels_mine, km.labels_):.4f}")
print(f"Agreement, from scratch vs true labels:  {agreement(labels_mine, y):.4f}")

fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
for ax, colour, title in [(axes[0], labels_mine, "k-means from scratch (k = 2)"),
                          (axes[1], y, "true labels (0 = malignant, 1 = benign)")]:
    ax.scatter(Z_svd[:, 0], Z_svd[:, 1], c=colour, cmap="coolwarm", alpha=0.6, s=15)
    ax.set_xlabel("PC1")
    ax.set_ylabel("PC2")
    ax.set_title(title)
plt.show()

*Your answer here.*